# 如何处理速率限制

当你反复调用 OpenAI API 时，可能会遇到 `429: 'Too Many Requests'` 或 `RateLimitError` 之类的错误消息。这些错误消息是由于超出了 API 的速率限制而产生的。

本指南分享了避免和处理速率限制错误的技巧。

如需查看一个通过限流并行请求来避免速率限制错误的示例脚本，请参阅 [api_request_parallel_processor.py](https://github.com/openai/openai-cookbook/blob/main/examples/api_request_parallel_processor.py)。

## 为什么存在速率限制

速率限制是 API 的常见做法，设置它出于以下几个原因。

- 首先，它们有助于防止 API 被滥用或误用。例如，恶意行为者可能会用大量请求淹没 API，试图使其过载或导致服务中断。通过设置速率限制，OpenAI 可以防止此类行为。
- 其次，速率限制有助于确保每个人都能公平地访问 API。如果某个人或组织发出过多的请求，可能会拖慢其他所有人的 API 使用。通过限制单个用户可以发出的请求数量，OpenAI 确保每个人都有机会使用 API，而不会遇到速度变慢的问题。
- 最后，速率限制可以帮助 OpenAI 管理其基础设施的总体负载。如果对 API 的请求急剧增加，可能会给服务器带来压力并导致性能问题。通过设置速率限制，OpenAI 可以帮助所有用户保持流畅一致的体验。

虽然触及速率限制可能令人沮丧，但速率限制的存在是为了保障 API 为其用户可靠运行。

## 默认速率限制

你的速率限制和支出限额（配额）会根据多种因素自动调整。随着你对 OpenAI API 的使用量增加并成功支付账单，我们会自动提升你的使用等级。你可以通过以下资源找到有关速率限制的具体信息。

### 其他速率限制资源

在以下资源中进一步了解 OpenAI 的速率限制：

- [指南：速率限制](https://platform.openai.com/docs/guides/rate-limits?context=tier-free)
- [帮助中心：API 使用是否受速率限制？](https://help.openai.com/en/articles/5955598-is-api-usage-subject-to-any-rate-limits)
- [帮助中心：如何解决 429: 'Too Many Requests' 错误？](https://help.openai.com/en/articles/5955604-how-can-i-solve-429-too-many-requests-errors)

### 申请提高速率限制

要进一步了解如何提高组织的使用等级和速率限制，请访问你的 [Limits 设置页面](https://platform.openai.com/account/limits)。

In [3]:
import openai
import os

client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY", "<your OpenAI API key if not set as env var>"))

## 速率限制错误示例

当 API 请求发送得过快时，就会发生速率限制错误。如果使用 OpenAI Python 库，错误看起来大致如下：

```
RateLimitError: Rate limit reached for default-codex in organization org-{id} on requests per min. Limit: 20.000000 / min. Current: 24.000000 / min. Contact support@openai.com if you continue to have issues or if you’d like to request an increase.
```

下面是触发速率限制错误的示例代码。

In [2]:
# request a bunch of completions in a loop
for _ in range(100):
    client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": "Hello"}],
        max_tokens=10,
    )

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

## 如何缓解速率限制错误

### 使用指数退避重试

缓解速率限制错误的一种简单方法是使用随机指数退避自动重试请求。使用指数退避重试意味着在遇到速率限制错误时先短暂休眠，然后重试失败的请求。如果请求仍然失败，则增加休眠时长并重复这一过程。如此反复，直到请求成功或达到最大重试次数为止。

这种方法有很多好处：

- 自动重试意味着你可以从速率限制错误中恢复，而不会崩溃或丢失数据
- 指数退避意味着最初的几次重试可以快速进行，而如果前几次重试失败，仍然可以受益于更长的延迟
- 在延迟中加入随机抖动（jitter）有助于避免所有重试同时发生

请注意，失败的请求也会计入你的每分钟限额，因此不断重新发送请求是行不通的。

下面是几个示例解决方案。

#### 示例 #1：使用 Tenacity 库

[Tenacity](https://tenacity.readthedocs.io/en/latest/) 是一个采用 Apache 2.0 许可证、用 Python 编写的通用重试库，可以简化为几乎任何操作添加重试行为的工作。

要为请求添加指数退避，你可以使用 `tenacity.retry` [装饰器](https://peps.python.org/pep-0318/)。下面的示例使用 `tenacity.wait_random_exponential` 函数为请求添加随机指数退避。

请注意，Tenacity 库是第三方工具，OpenAI 不对其可靠性或安全性做任何保证。

In [6]:
from tenacity import (
    retry,
    stop_after_attempt,
    wait_random_exponential,
)  # for exponential backoff

@retry(wait=wait_random_exponential(min=1, max=60), stop=stop_after_attempt(6))
def completion_with_backoff(**kwargs):
    return client.chat.completions.create(**kwargs)


completion_with_backoff(model="gpt-4o-mini", messages=[{"role": "user", "content": "Once upon a time,"}])

ChatCompletion(id='chatcmpl-8PAu6anX2JxQdYmJRzps38R8u0ZBC', choices=[Choice(finish_reason='stop', index=0, message=ChatCompletionMessage(content='in a small village nestled among green fields and rolling hills, there lived a kind-hearted and curious young girl named Lily. Lily was known for her bright smile and infectious laughter, bringing joy to everyone around her.\n\nOne sunny morning, as Lily played in the meadows, she stumbled upon a mysterious book tucked away beneath a tall oak tree. Intrigued, she picked it up and dusted off its weathered cover to reveal intricate golden patterns. Without hesitation, she opened it, discovering that its pages were filled with magical tales and enchanting adventures.\n\nAmong the stories she found, one particularly caught her attention—a tale of a long-lost treasure hidden deep within a mysterious forest. Legend had it that whoever found this hidden treasure would be granted one wish, no matter how big or small. Excited by the prospect of findin

#### 示例 #2：使用 backoff 库

另一个提供退避和重试函数装饰器的库是 [backoff](https://pypi.org/project/backoff/)。

与 Tenacity 一样，backoff 库也是第三方工具，OpenAI 不对其可靠性或安全性做任何保证。

In [ ]:
import backoff  # for exponential backoff

@backoff.on_exception(backoff.expo, openai.RateLimitError, max_time=60, max_tries=6)
def completions_with_backoff(**kwargs):
    return client.chat.completions.create(**kwargs)


completions_with_backoff(model="gpt-4o-mini", messages=[{"role": "user", "content": "Once upon a time,"}])


ChatCompletion(id='chatcmpl-AqRiD3gF3q8VVs6w8jgba6FHGr0L5', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="in a small village nestled between lush green hills and a shimmering lake, there lived a young girl named Elara. Elara had a curious spirit and a heart full of dreams. Every day, she would explore the woods surrounding her home, searching for hidden treasures and magical creatures.\n\nOne sunny afternoon, while wandering deeper into the forest than she ever had before, Elara stumbled upon a sparkling, crystal-clear pond. As she knelt down to take a closer look, she noticed a glimmering object at the bottom. It was a beautifully crafted key, shining with an otherworldly light. Without thinking twice, Elara reached into the cool water and retrieved the key, feeling a strange warmth envelop her.\n\nLittle did she know, this key was no ordinary key. It was said to unlock a secret door hidden in the heart of the forest, a door that 

#### 示例 3：手动实现退避

如果你不想使用第三方库，也可以实现自己的退避逻辑。

In [11]:
# imports
import random
import time

# define a retry decorator
def retry_with_exponential_backoff(
    func,
    initial_delay: float = 1,
    exponential_base: float = 2,
    jitter: bool = True,
    max_retries: int = 10,
    errors: tuple = (openai.RateLimitError,),
):
    """Retry a function with exponential backoff."""

    def wrapper(*args, **kwargs):
        # Initialize variables
        num_retries = 0
        delay = initial_delay

        # Loop until a successful response or max_retries is hit or an exception is raised
        while True:
            try:
                return func(*args, **kwargs)

            # Retry on specified errors
            except errors as e:
                # Increment retries
                num_retries += 1

                # Check if max retries has been reached
                if num_retries > max_retries:
                    raise Exception(
                        f"Maximum number of retries ({max_retries}) exceeded."
                    )

                # Increment the delay
                delay *= exponential_base * (1 + jitter * random.random())

                # Sleep for the delay
                time.sleep(delay)

            # Raise exceptions for any errors not specified
            except Exception as e:
                raise e

    return wrapper


@retry_with_exponential_backoff
def completions_with_backoff(**kwargs):
    return client.chat.completions.create(**kwargs)


completions_with_backoff(model="gpt-4o-mini", messages=[{"role": "user", "content": "Once upon a time,"}])

ChatCompletion(id='chatcmpl-8PAxGvV3GbLpnOoKSvJ00XCUdOglM', choices=[Choice(finish_reason='stop', index=0, message=ChatCompletionMessage(content="in a faraway kingdom, there lived a young princess named Aurora. She was known for her beauty, grace, and kind heart. Aurora's kingdom was filled with lush green meadows, towering mountains, and sparkling rivers. The princess loved spending time exploring the enchanting forests surrounding her castle.\n\nOne day, while Aurora was wandering through the woods, she stumbled upon a hidden clearing. At the center stood a majestic oak tree, its branches reaching towards the sky. Aurora approached the tree with curiosity, and as she got closer, she noticed a small door at its base.\n\nIntrigued, she gently pushed open the door and was amazed to find herself in a magical realm. The forest transformed into a breathtaking wonderland, with colorful flowers blooming in every direction and woodland creatures frolicking joyously. Aurora's eyes widened with

### 回退到另一个模型

如果你在主模型上遇到速率限制错误，一种选择是切换到备用模型。当主模型被限流或不可用时，这种方法有助于保持应用的响应能力。

然而，备用模型在准确性、延迟和成本方面可能存在显著差异。因此，这种策略可能并不适用于所有用例，尤其是那些需要高度一致结果的用例。此外，请记住，某些模型共享速率限制，这可能会降低简单切换模型的效果。你可以在[组织限额页面](https://platform.openai.com/settings/organization/limits)中查看共享限额的模型。

在将这种方法部署到生产环境之前，请彻底测试它对输出质量、用户体验和运营预算的影响。使用相关评估来验证你的回退方案，确保它满足你的要求，并在真实环境中保持可接受的性能。

In [7]:
def completions_with_fallback(fallback_model, **kwargs):
    try:
        return client.chat.completions.create(**kwargs)
    except openai.RateLimitError:
        kwargs['model'] = fallback_model
        return client.chat.completions.create(**kwargs)
    
    
completions_with_fallback(fallback_model="gpt-4o", model="gpt-4o-mini", messages=[{"role": "user", "content": "Once upon a time,"}])

ChatCompletion(id='chatcmpl-AsX9Zts2toXoKA80ZujeWMXKMolBy', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='in a quaint little village nestled between lush green hills and sparkling blue rivers, there lived a young girl named Elara. Elara was known for her adventurous spirit and her unwavering curiosity about the world beyond her village. She often spent her days wandering the meadows, exploring the enchanted forest, and collecting wildflowers.\n\nOne sunny afternoon, while she was picking daisies near the edge of the forest, Elara stumbled upon an old, ornate key half-buried in the ground. Intrigued, she dusted it off and inspected it closely. The key was beautifully crafted, with intricate patterns carved into its metal. Elara felt a strange pull towards it, as if it were meant for her.\n\nDetermined to uncover its secrets, Elara ran back to the village, her heart racing with excitement. She gathered her closest friends—Jasper, a c

### 减小 `max_tokens` 以匹配预期的补全长度

速率限制的用量按以下两者中较大的一个计算：
1. `max_tokens` - 响应中允许的最大 token 数。
2. 输入中的估算 token 数——根据提示的字符数推算得出。

如果你将 `max_tokens` 设置得过高，即使实际响应要短得多，你的用量也可能被高估。为避免过早触及速率限制，请将 `max_tokens` 配置为与预期响应的大小接近。这可以确保更准确的用量计算，并有助于防止意外的限流。

In [11]:
def completions_with_max_tokens(**kwargs):
    return client.chat.completions.create(**kwargs)


completions_with_max_tokens(model="gpt-4o-mini", messages=[{"role": "user", "content": "Once upon a time,"}], max_tokens=100)

ChatCompletion(id='chatcmpl-Aq0JmjugPw2i232ZEZuK5inHnx6Vc', choices=[Choice(finish_reason='length', index=0, logprobs=None, message=ChatCompletionMessage(content='in a small village nestled between lush green hills and a sparkling river, there lived a young girl named Lila. Lila was known for her boundless curiosity and adventurous spirit. She had a wild imagination, often spinning tales about the mysteries that lay beyond the village borders.\n\nOne day, while exploring the forest, Lila stumbled upon a hidden path she had never seen before. The path was winding and overgrown, beckoning her with whispers of adventure. Against her better judgment, she decided to', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None), internal_metrics=[{'cached_prompt_tokens': 0, 'total_accepted_tokens': 0, 'total_batched_tokens': 127, 'total_predicted_tokens': 0, 'total_rejected_tokens': 0, 'total_tokens_in_completion': 128, 'cached_embeddings_bytes': 0, 'cached_embeddings_n'

## 如何在速率限制下最大化批处理吞吐量

如果你在处理来自用户的实时请求，退避和重试是在避免速率限制错误的同时最小化延迟的绝佳策略。

然而，如果你在处理大量批量数据，吞吐量比延迟更重要，那么除了退避和重试之外，你还可以做一些其他事情。

### 主动在请求之间添加延迟

如果你不断地触及速率限制、退避、再次触及速率限制、再次退避，那么很可能有相当一部分请求预算被“浪费”在需要重试的请求上。在速率限制固定的情况下，这会限制你的处理吞吐量。

一种可能的解决方案是计算你的速率限制，并添加与其倒数相等的延迟（例如，如果你的速率限制是每分钟 20 个请求，则为每个请求添加 3–6 秒的延迟）。这可以帮助你在接近速率限制上限的情况下运行，而不会触及上限并产生浪费的请求。

#### 为请求添加延迟的示例

In [12]:
import time

# Define a function that adds a delay to a Completion API call
def delayed_completion(delay_in_seconds: float = 1, **kwargs):
    """Delay a completion by a specified amount of time."""

    # Sleep for the delay
    time.sleep(delay_in_seconds)

    # Call the Completion API and return the result
    return client.chat.completions.create(**kwargs)


# Calculate the delay based on your rate limit
rate_limit_per_minute = 20
delay = 60.0 / rate_limit_per_minute

delayed_completion(
    delay_in_seconds=delay,
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Once upon a time,"}]
)


ChatCompletion(id='chatcmpl-8PAyCR1axKsomV0e349XiCN1Z81pH', choices=[Choice(finish_reason='stop', index=0, message=ChatCompletionMessage(content="in a small village, there lived a young girl named Maya. Maya was known for her kindness and love for nature. She spent hours exploring the forests surrounding the village, admiring the vibrant flowers and talking to the animals.\n\nOne sunny day, as Maya was picking wildflowers, she stumbled upon a wounded blackbird with a broken wing. Feeling sorry for the bird, Maya gently picked it up and cradled it in her hands. She knew she had to help the bird, so she hurried back to her cottage.\n\nMaya set up a cozy nest for the blackbird and carefully splinted its wing. She fed it worms and berries, doing everything she could to nurse it back to health. Each day, she would sing lullabies and tell stories to keep the blackbird company. Slowly, the bird's wing healed, and before long, it was ready to fly again.\n\nOn a beautiful morning, Maya opened t

### 批量请求

OpenAI API 对每分钟/每天请求数（RPM/RPD）和每分钟 token 数（TPM）分别实施限制。如果你触及了 RPM 限制，但仍有可用的 TPM 容量，可以考虑将多个任务合并到每个请求中。

通过将多个提示打包在一起，你可以减少每分钟发送的请求总数，从而有助于避免触及 RPM 上限。如果你谨慎管理 TPM 用量，这种方法还可能带来更高的整体吞吐量。不过，请记住以下几点：
- 每个模型在单个请求中可以处理的 token 数都有上限。如果批量提示超过此限制，请求将失败或被截断。
- 如果任务需要等待被合并到单个请求中，批处理可能会引入额外的等待时间。这可能会影响对时间敏感的应用的用户体验。
- 发送多个提示时，响应对象返回的顺序或格式可能与提交的提示不一致。你应该尝试通过对输出进行后处理，将每个响应与其对应的提示匹配起来。

#### 不使用批处理的示例

In [27]:
num_stories = 10
content = "Once upon a time,"

# serial example, with one story completion per request
for _ in range(num_stories):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": content}],
        max_tokens=20,
    )

    print(content + response.choices[0].message.content)


Once upon a time,in a quaint little village nestled between rolling hills and a sparkling river, there lived a young girl named
Once upon a time,Once upon a time, in a tranquil village nestled between rolling hills and lush forests, there lived a
Once upon a time,in a lush, green valley surrounded by towering mountains, there lay a small village called Eldergrove
Once upon a time,in a quaint little village nestled between rolling hills and a sparkling river, there lived a young girl named
Once upon a time,in a small village nestled between whispering woods and a sparkling river, there lived a curious young girl
Once upon a time,in a small village nestled between a vast forest and a shimmering lake, there lived a kind-hearted girl
Once upon a time,in a quaint little village nestled between rolling hills and a shimmering lake, there lived a curious girl named
Once upon a time,in a quaint little village nestled between emerald hills and a sparkling brook, there was a curious child named
O

#### 使用结构化输出在单个请求中批量处理多个提示的示例

OpenAI 的 [结构化输出（Structured Outputs）](https://platform.openai.com/docs/guides/structured-outputs) 功能让你可以在一个请求中批量处理多个提示。

与其解析原始文本或依赖非正式的格式，不如指定一个严格的 schema。结构化输出会强制执行该结构，从而减少解析和结构校验的工作。应用代码仍然必须校验业务规则以及每个值的含义。

In [6]:
from pydantic import BaseModel

# Define the Pydantic model for the structured output
class StoryResponse(BaseModel):
    stories: list[str]
    story_count: int

num_stories = 10
content = "Once upon a time,"

prompt_lines = [f"Story #{i+1}: {content}" for i in range(num_stories)]
prompt_text = "\n".join(prompt_lines)

messages = [
    {
        "role": "developer",
        "content": "You are a helpful assistant. Please respond to each prompt as a separate short story."
    },
    {
        "role": "user",
        "content": prompt_text
    }
]

# batched example, with all story completions in one request and using structured outputs
response = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=messages,
    response_format=StoryResponse,
)

print(response.choices[0].message.content)

{"stories":["Once upon a time, in a lush green valley, there lived a curious little fox named Felix. Every day, he would explore the woods, finding hidden glades and sparkling streams. One day, while chasing a butterfly, he stumbled upon a magical oak tree that granted wishes. Felix wished for courage and became the bravest fox in the land, helping his friends as they faced challenges together.","Once upon a time, a village known for its beautiful gardens fell into despair when a drought struck. The villagers prayed for rain but to no avail. One evening, a wise old woman arrived and told them of a hidden spring deep in the forest. With hope, the villagers embarked on a quest to find it, learning the value of teamwork and perseverance. Eventually, they found the spring, and the rain returned, reviving their gardens and spirits.","Once upon a time, in a kingdom high atop the clouds, lived Princess Lumina who had the ability to control the stars. But she felt lonely and longed for a compa

## 并行处理脚本示例

我们编写了一个用于并行处理大量 API 请求的示例脚本：[api_request_parallel_processor.py](https://github.com/openai/openai-cookbook/blob/main/examples/api_request_parallel_processor.py)。

该脚本结合了一些实用的特性：
- 从文件中流式读取请求，避免大型任务耗尽内存
- 并发发出请求，以最大化吞吐量
- 同时限制请求数和 token 用量，以保持在速率限制之下
- 重试失败的请求，以避免数据丢失
- 记录错误，以便诊断请求中的问题

你可以直接使用它，也可以根据需要进行修改。